# The Perceptron: Weight Updates, Logic Gates, and the XOR Problem

##  The Weight Update Formula

The "learning" in a Perceptron is simply the process of adjusting the weights to minimize error. This is done using the **Perceptron Learning Rule**.

### The Mathematical Formula
When an error occurs, we update the weights and bias using:

$$w_{new} = w_{old} + \Delta w$$
$$b_{new} = b_{old} + \Delta b$$

Where the change ($\Delta$) is calculated as:

$$\Delta w = \eta \cdot (y - \hat{y}) \cdot x_i$$
$$\Delta b = \eta \cdot (y - \hat{y})$$

**Key Variables:**
* **$\eta$ (Eta):** The **Learning Rate** (e.g., 0.1). It controls how big a "step" we take.
* **$y$:** The **Actual Target** (Ground Truth).
* **$\hat{y}$:** The **Predicted Output** (Perceptron's guess).
* **$x_i$:** The **Input value** associated with that weight.
* **$(y - \hat{y})$:** The **Error Term**.

---

## Weight Update Intuition

**What is actually happening geometrically?**

The Perceptron is trying to draw a line (a decision boundary) to separate two classes of data. The weight vector ($w$) points perpendicular to this line.



### Scenario A: False Negative (Target=1, Prediction=0)
* **Error:** $(1 - 0) = +1$.
* **Action:** We **ADD** the input vector to the weight vector.
* **Intuition:** This rotates the weight vector *towards* the input $x$, making the Perceptron more likely to fire (output 1) for this input next time.

### Scenario B: False Positive (Target=0, Prediction=1)
* **Error:** $(0 - 1) = -1$.
* **Action:** We **SUBTRACT** the input vector from the weight vector.
* **Intuition:** This pushes the weight vector *away* from the input $x$, making it less likely to fire (output 1) next time.

---

## Solving the Logical AND Gate

Let's trace how a Perceptron learns the **AND** function manually.

### Part 1: The Setup
**Truth Table for AND:**
| $x_1$ | $x_2$ | Target ($y$) |
| :--- | :--- | :--- |
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

* **Initial Weights:** $[0.0, 0.0]$
* **Bias:** $0.0$
* **Learning Rate:** $0.1$
* **Activation:** Step Function ($1$ if $z \ge 0$, else $0$).

### Part 2: The Trace (First Few Steps)
**Step 1: Input (1, 1), Target 1**
* $z = (1 \cdot 0) + (1 \cdot 0) + 0 = 0$.
* Prediction $\hat{y} = 1$ (since $z \ge 0$). **Correct.** (No update).

**Step 2: Input (0, 0), Target 0**
* $z = 0$. Prediction $\hat{y} = 1$. **ERROR.**
* Error = $0 - 1 = -1$.
* $\Delta w = 0.1 \cdot (-1) \cdot 0 = 0$. (Weights don't change).
* $\Delta b = 0.1 \cdot (-1) = -0.1$.
* **New Bias:** $-0.1$.

**Step 3: Input (0, 1), Target 0**
* $z = (0 \cdot 0) + (1 \cdot 0) - 0.1 = -0.1$.
* Prediction $\hat{y} = 0$. **Correct.**

*The Perceptron continues looping until it finds weights like $w=[0.5, 0.5]$ and $b=-0.7$.*
* Check (1,1): $0.5+0.5 - 0.7 = 0.3 \rightarrow 1$ (Correct).
* Check (0,1): $0+0.5 - 0.7 = -0.2 \rightarrow 0$ (Correct).



---


In [ ]:


import numpy as np

class Perceptron:
    def __init__(self, learning_rate=0.1, n_iters=10):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def activation(self, z):
        # Heaviside Step Function
        return 1 if z >= 0 else 0

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Init parameters
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Training Loop
        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                # 1. Forward Pass
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = self.activation(linear_output)
                
                # 2. Perceptron Update Rule
                update = self.lr * (y[idx] - y_predicted)
                
                self.weights += update * x_i
                self.bias += update

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return [self.activation(i) for i in linear_output]

In [ ]:
# logic_gates.py

# 1. Define Data (AND Gate)
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y_and = np.array([0, 0, 0, 1])

# 2. Train
p = Perceptron(learning_rate=0.1, n_iters=10)
p.fit(X, y_and)

# 3. Test
print("AND Gate Weights:", p.weights)
print("AND Gate Bias:", p.bias)
print("Predictions:", p.predict(X)) 
# Output should be [0, 0, 0, 1]

## The XOR Problem (Why it Fails)
The OR gate is easy (similar to AND). The XOR (Exclusive OR) gate is impossible for a single Perceptron.

## Why it Fails: Linear Separability
A Single Layer Perceptron is a Linear Classifier. It can only draw a straight line to separate data.

Look at the graph:

The "0"s are at (0,0) and (1,1).

The "1"s are at (0,1) and (1,0).

**The Geometry:**

Try to draw a single straight line that puts the "0"s on one side and the "1"s on the other.

It is mathematically impossible.

**The Result**:

The Perceptron's weights will oscillate endlessly, never settling on a solution. The error will never reach 0.

### The Solution: Multi-Layer Perceptron (MLP)
To solve XOR, you need Non-Linearity. By adding a Hidden Layer, you essentially combine two lines. One neuron handles the "OR" logic, another handles the "AND" logic, and they combine to create an XOR boundary.